# Cleaning GLAMOS Flow Velocity 2025 — corrected column alignment

This notebook works with `flow_velocity_clean.csv`: semicolon-delimited, but with a one-column shift in the header relative to the data starting at column 16.

**Column layout (from the units row):**

| pos | true meaning | unit |
|---|---|---|
| 12 | `d_t` | decimal days |
| 13 | `d_x` | m |
| 14 | `d_y` | m |
| 15 | `d_z` | m |
| 16 | `ablation_m` (surface elevation change at the stake from melt/accumulation) | m |
| 17 | `velocity_xy` | m/a |
| 18 | `velocity_z` | m/a |

**Verified formulas**

- `velocity_xy = sqrt(d_x² + d_y²) / (d_t / 365.25)`
- `velocity_z  = d_z / (d_t / 365.25)`

With those, the only column we **can't** impute is `ablation_m` itself — it's an independent stake reading, not derivable from the other fields.

Source: GLAMOS (2025), `doi:10.18750/flowvelocity.2025.r2025`.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

## 1. Load with the corrected column names

Skip 7 preamble lines + the (shifted) header row + the units row, then assign column names by position.

In [4]:
CSV_PATH = Path("../docs/data/csv/flow_velocity_clean.csv")  # adjust if the file lives elsewhere

# Correct column names, by position, based on the units row.
# Positions 1-18 are unambiguous; positions 19+ are free-text fields that may contain ';' so we don't rely on them being clean.
COL_NAMES = [
    "stake_name", "glacier_name", "SGI_ID", "WGMS_ID",        # 1-4
    "date_from", "time_from", "date_to", "time_to",            # 5-8
    "latitude_from", "longitude_from", "altitude_from",         # 9-11
    "d_t", "d_x", "d_y", "d_z",                                # 12-15
    "ablation_m",                                                # 16  (mystery column, units m)
    "velocity_xy", "velocity_z",                                # 17-18
]

PREAMBLE_LINES = 9  # 7 preamble + 1 header + 1 units row

def load_clean(path: Path) -> pd.DataFrame:
    rows = []
    with open(path, "r", encoding="utf-8-sig") as f:
        for i, line in enumerate(f, start=1):
            if i <= PREAMBLE_LINES:
                continue
            line = line.rstrip("\r\n")
            if not line.strip():
                continue
            parts = line.split(";")
            if len(parts) < 18:
                continue
            rows.append(parts[:18])

    df = pd.DataFrame(rows, columns=COL_NAMES)

    numeric_cols = ["latitude_from", "longitude_from", "altitude_from",
                    "d_t", "d_x", "d_y", "d_z", "ablation_m",
                    "velocity_xy", "velocity_z"]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["date_from"] = pd.to_datetime(df["date_from"], errors="coerce")
    df["date_to"]   = pd.to_datetime(df["date_to"],   errors="coerce")

    for c in df.select_dtypes(include=["object", "string"]).columns:
        df[c] = df[c].replace(r"^\s*$", np.nan, regex=True)

    return df

df_raw = load_clean(CSV_PATH)
print(f"Loaded {len(df_raw)} rows, {df_raw.shape[1]} columns")
df_raw.head()

Loaded 2451 rows, 18 columns


,stake_name,glacier_name,SGI_ID,WGMS_ID,date_from,time_from,date_to,time_to,latitude_from,longitude_from,altitude_from,d_t,d_x,d_y,d_z,ablation_m,velocity_xy,velocity_z
0,0306,Silvrettagletscher,A10g-05,408,2003-09-21,11:04:00,2004-09-21,13:31:00,800515.31,192889.65,2628.11,366.1,-5.96,-0.24,-0.28,-0.53,5.95,-0.28
1,0308,Silvrettagletscher,A10g-05,408,2003-09-21,16:35:00,2004-09-21,14:31:00,799827.45,192744.79,2538.54,365.9,-3.97,-0.42,-1.11,-1.28,3.99,-1.11
2,0312,Silvrettagletscher,A10g-05,408,2003-09-21,14:02:00,2004-09-21,13:03:00,800406.17,192586.92,2603.59,366.0,-6.06,1.09,-0.92,-0.47,6.14,-0.92
3,0311,Silvrettagletscher,A10g-05,408,2003-09-21,15:04:00,2004-09-21,12:31:00,800718.03,192205.65,2727.24,365.9,-1.30,1.36,-0.47,-0.52,1.88,-0.47
4,0307,Silvrettagletscher,A10g-05,408,2003-09-21,12:03:00,2004-09-21,14:01:00,800156.96,192867.07,2581.57,366.1,-3.88,-1.23,-0.46,NaN,4.06,-0.46


## 2. Missing-value overview

In [6]:
missing_summary = pd.DataFrame({
    "n_missing":   df_raw.isna().sum(),
    "pct_missing": (df_raw.isna().mean() * 100).round(2),
    "dtype":       df_raw.dtypes,
})
missing_summary

,n_missing,pct_missing,dtype
stake_name,0,0.00,object
glacier_name,0,0.00,object
SGI_ID,0,0.00,object
WGMS_ID,0,0.00,object
date_from,0,0.00,datetime64[ns]
time_from,0,0.00,object
date_to,0,0.00,datetime64[ns]
time_to,0,0.00,object
latitude_from,0,0.00,float64
longitude_from,0,0.00,float64


## 3. Impute what we can

Three relationships, all using the verified formulas:

1. `velocity_xy` ← `d_x`, `d_y`, `d_t`
2. `velocity_z`  ← `d_z`, `d_t`
3. `d_z`         ← `velocity_z`, `d_t`  *(single component, fully invertible)*

`d_x` and `d_y` can't be recovered from `velocity_xy` alone — that's the magnitude of two components; you'd need a flow direction.

`ablation_m` is an independent stake reading and can't be derived from the others.

In [9]:
def impute(df, checks):
    df = df.copy()
    report = {}
    yrs = years_elapsed(df["d_t"])

    if checks.get("vxy_from_dxdy", {}).get("fits"):
        mask = df["velocity_xy"].isna() & df["d_x"].notna() & df["d_y"].notna() & df["d_t"].notna()
        df.loc[mask, "velocity_xy"] = (
            np.sqrt(df.loc[mask, "d_x"] ** 2 + df.loc[mask, "d_y"] ** 2) / yrs[mask]
        )
        report["velocity_xy filled"] = int(mask.sum())
    else:
        report["velocity_xy filled"] = "skipped — formula check failed"

    if checks.get("vz_from_dz", {}).get("fits"):
        mask = df["velocity_z"].isna() & df["d_z"].notna() & df["d_t"].notna()
        df.loc[mask, "velocity_z"] = df.loc[mask, "d_z"] / yrs[mask]
        report["velocity_z filled"] = int(mask.sum())

        mask = df["d_z"].isna() & df["velocity_z"].notna() & df["d_t"].notna()
        df.loc[mask, "d_z"] = df.loc[mask, "velocity_z"] * yrs[mask]
        report["d_z filled"] = int(mask.sum())
    else:
        report["velocity_z filled"] = "skipped — formula check failed"
        report["d_z filled"]        = "skipped — formula check failed"

    report["d_x / d_y"]    = "not recoverable from velocity_xy alone (magnitude only)"
    report["ablation_m"]   = "not recoverable (independent measurement)"
    return df, report

df_imputed, impute_report = impute(df_raw, checks)
impute_report

{'velocity_xy filled': 0,
 'velocity_z filled': 0,
 'd_z filled': 122,
 'd_x / d_y': 'not recoverable from velocity_xy alone (magnitude only)',
 'ablation_m': 'not recoverable (independent measurement)'}

In [10]:
numeric_target_cols = ["d_x", "d_y", "d_z", "ablation_m", "velocity_xy", "velocity_z"]
before = df_raw[numeric_target_cols].isna().sum()
after  = df_imputed[numeric_target_cols].isna().sum()
pd.DataFrame({"before": before, "after": after, "recovered": before - after})

,before,after,recovered
d_x,1013,1013,0
d_y,1013,1013,0
d_z,1038,916,122
ablation_m,444,444,0
velocity_xy,37,37,0
velocity_z,916,916,0


## 4. Two output datasets

- **`flowvelocity_2025_imputed.csv`** — rows with all numeric columns filled (after imputation). Use this if you need complete velocity records.
- **`flowvelocity_2025_imputed_no_ablation.csv`** — same imputation, but only requires `d_x`, `d_y`, `d_z`, `velocity_xy`, `velocity_z` to be present. Keeps more rows for flow-velocity-only analyses where `ablation_m` isn't needed.

In [13]:
OUTPUT_DIR = Path("../docs/data/csv/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

vel_cols = ["d_x", "d_y", "d_z", "velocity_xy", "velocity_z"]

df_full      = df_imputed.dropna(subset=numeric_target_cols).copy()
df_vel_only  = df_imputed.dropna(subset=vel_cols).copy()

df_full.to_csv(OUTPUT_DIR / "flowvelocity_2025_imputed.csv", index=False)
df_vel_only.to_csv(OUTPUT_DIR / "flowvelocity_2025_imputed_no_ablation.csv", index=False)

print(f"Total rows:                              {len(df_raw)}")
print(f"Imputed + all numeric cols present:      {len(df_full)}  ({len(df_full)/len(df_raw):.1%})")
print(f"Imputed + velocity/displacement present: {len(df_vel_only)}  ({len(df_vel_only)/len(df_raw):.1%})")

Total rows:                              2451
Imputed + all numeric cols present:      1074  (43.8%)
Imputed + velocity/displacement present: 1413  (57.6%)


In [12]:
df_vel_only.head()

,stake_name,glacier_name,SGI_ID,WGMS_ID,date_from,time_from,date_to,time_to,latitude_from,longitude_from,altitude_from,d_t,d_x,d_y,d_z,ablation_m,velocity_xy,velocity_z
0,0306,Silvrettagletscher,A10g-05,408,2003-09-21,11:04:00,2004-09-21,13:31:00,800515.31,192889.65,2628.11,366.1,-5.96,-0.24,-0.28,-0.53,5.95,-0.28
1,0308,Silvrettagletscher,A10g-05,408,2003-09-21,16:35:00,2004-09-21,14:31:00,799827.45,192744.79,2538.54,365.9,-3.97,-0.42,-1.11,-1.28,3.99,-1.11
2,0312,Silvrettagletscher,A10g-05,408,2003-09-21,14:02:00,2004-09-21,13:03:00,800406.17,192586.92,2603.59,366.0,-6.06,1.09,-0.92,-0.47,6.14,-0.92
3,0311,Silvrettagletscher,A10g-05,408,2003-09-21,15:04:00,2004-09-21,12:31:00,800718.03,192205.65,2727.24,365.9,-1.30,1.36,-0.47,-0.52,1.88,-0.47
4,0307,Silvrettagletscher,A10g-05,408,2003-09-21,12:03:00,2004-09-21,14:01:00,800156.96,192867.07,2581.57,366.1,-3.88,-1.23,-0.46,NaN,4.06,-0.46
